In [1]:
import sys
from pyspark.sql import SparkSession
from awsglue.context import GlueContext
from pyspark.context import SparkContext
from pyspark.sql.functions import current_timestamp, lit

# Initialize Spark and Glue Context
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

SOURCE_BUCKET = "formulaone-glue"
FOLDER = "landing"
file_name_audit = "circuits.csv"

# 1. Load the data FIRST
# Use .option("inferSchema", "true") if you want Spark to guess data types (int, double, etc.)
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(f"s3://{SOURCE_BUCKET}/{FOLDER}/circuits.csv")

# 2. Define the rename mapping
rename_map = {
    "circuitId": "circuit_id",
    "circuitRef": "circuit_ref",
    "lat": "latitude",
    "lng": "longitude",
    "alt": "altitude",
}

# 3. Apply renames and add metadata
for old_name, new_name in rename_map.items():
    if old_name in df.columns:
        df = df.withColumnRenamed(old_name, new_name)

# Adding ingestion timestamp (Standard practice in Glue ETL)
df = df.withColumn("ingestion_timestamp", current_timestamp()).withColumn("file_name", lit(file_name_audit))

# 4. Register as a temporary view
df.createOrReplaceTempView("formulaone_circuits_csv")

# 5. Query and Display
spark.sql("""SELECT 
             circuit_id,
             circuit_ref,
             name,
             location,
             country,
             latitude,
             longitude,
             altitude,
             ingestion_timestamp,
             file_name
             FROM formulaone_circuits_csv""").show(10, truncate=False)

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
0,None,pyspark,idle,,,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----------+--------------+------------------------------+------------+---------+--------+---------+--------+-----------------------+------------+
|circuit_id|circuit_ref   |name                          |location    |country  |latitude|longitude|altitude|ingestion_timestamp    |file_name   |
+----------+--------------+------------------------------+------------+---------+--------+---------+--------+-----------------------+------------+
|1         |albert_park   |Albert Park Grand Prix Circuit|Melbourne   |Australia|-37.8497|144.968  |10      |2026-04-09 15:21:41.553|circuits.csv|
|2         |sepang        |Sepang International Circuit  |Kuala Lumpur|Malaysia |2.76083 |101.738  |18      |2026-04-09 15:21:41.553|circuits.csv|
|3         |bahrain       |Bahrain International Circuit |Sakhir      |Bahrain  |26.0325 |50.5106  |7       |2026-04-09 15:21:41.553|circuits.csv|
|4         |catalunya     |Circuit de Barcelona-Catalunya|Montmeló    |Spain    |41.57   |2.26111  |109     |2026-04-0

In [15]:
import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from awsglue.dynamicframe import DynamicFrame
from pyspark.sql.functions import current_timestamp, lit

# Initialize Glue Context
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

# Configuration
SOURCE_BUCKET = "formulaone-glue"
FOLDER = "landing"
FILE_NAME = "circuits.csv"
S3_PATH = f"s3://{SOURCE_BUCKET}/{FOLDER}/{FILE_NAME}"

# 1. Load Data
# We treat everything as a string initially to prevent the parser from dropping data
dyf = glueContext.create_dynamic_frame.from_options(
    connection_type="s3",
    connection_options={"paths": [S3_PATH], "recurse": True},
    format="csv",
    format_options={
        "withHeader": False, 
        "optimizePerformance": True,
        "separator": ","
    }
)

# 2. Data Cleaning Function
# This replaces "\N" or empty strings with "0" (or you can leave it as None)
# so that the subsequent cast to Double/Long doesn't fail.
def clean_f1_data(rec):
    # col8 is altitude. If it's "\N", change it to "0" so it can be cast to double.
    if rec["col8"] == "\\N" or rec["col8"] == "":
        rec["col8"] = "0"
    return rec

dyf_cleaned = Map.apply(frame=dyf, f=clean_f1_data)

# 3. Resolve Data Type Ambiguity
# Now that "\N" is gone, 'cast:double' will work perfectly.
dyf_resolved = dyf_cleaned.resolveChoice(specs=[
    ("col0", "cast:long"),
    ("col6", "cast:double"),
    ("col7", "cast:double"),
    ("col8", "cast:double")
])

# 4. Apply Mapping
dyf_mapped = ApplyMapping.apply(
    frame=dyf_resolved,
    mappings=[
        ("col0", "long", "circuit_id", "long"),
        ("col1", "string", "circuit_ref", "string"),
        ("col2", "string", "name", "string"),
        ("col3", "string", "location", "string"),
        ("col4", "string", "country", "string"),
        ("col6", "double", "latitude", "double"),
        ("col7", "double", "longitude", "double"),
        ("col8", "double", "altitude", "double")
    ]
)

# 5. Final Spark DF Processing
# We filter out the header row (where col0 was the string "circuitId")
df_final = dyf_mapped.toDF().filter("circuit_id IS NOT NULL")

# Add Metadata
df_final = df_final.withColumn("ingestion_timestamp", current_timestamp()) \
                   .withColumn("file_name", lit(FILE_NAME))

# 6. Output and Verify
df_final.createOrReplaceTempView("formulaone_circuits")

spark.sql("""
    SELECT 
        circuit_id, 
        circuit_ref,
        name, 
        location,
        country,
        latitude, 
        longitude, 
        altitude, 
        ingestion_timestamp 
    FROM formulaone_circuits
    ORDER BY circuit_id ASC
""").show(100, truncate=False)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----------+--------------+-------------------------------------+---------------------+------------+---------+---------+--------+-----------------------+
|circuit_id|circuit_ref   |name                                 |location             |country     |latitude |longitude|altitude|ingestion_timestamp    |
+----------+--------------+-------------------------------------+---------------------+------------+---------+---------+--------+-----------------------+
|1         |albert_park   |Albert Park Grand Prix Circuit       |Melbourne            |Australia   |144.968  |10.0     |null    |2026-04-09 15:35:34.702|
|2         |sepang        |Sepang International Circuit         |Kuala Lumpur         |Malaysia    |101.738  |18.0     |null    |2026-04-09 15:35:34.702|
|3         |bahrain       |Bahrain International Circuit        |Sakhir               |Bahrain     |50.5106  |7.0      |null    |2026-04-09 15:35:34.702|
|4         |catalunya     |Circuit de Barcelona-Catalunya       |Montmeló   